# 05 - Stable baseline versus configured Python preview [PREVIEW]

**Hypothesis:** counting only retrievable candidates can align advertised pages, while increasing text recall can make more keyword candidates reachable.

Compare the existing stable Python evidence with two Python `azure-search-documents==12.1.0b2` scenarios using API `2026-08-01-preview` and `countRetrievableResults`: text recall **1,000**, then **6,158**. The stable SDK/notebook and .NET project stay unchanged. No infrastructure, index, or document writes occur here.

This intentionally shorter experiment changes SDK, API and settings together. It does **not** isolate an SDK-only fix, execute the .NET beta package, approve production use, or resolve the failed keyword complete-ID baseline.

Use the original demo Python kernel with `notebooks/` as the working directory. Preview dependencies live in a separate `.venv-preview`, invoked as a subprocess; no kernel switch is needed. Budget roughly 1,500 sequential reads if both counts align as expected, potentially more if the service reports different counts. Actual behavior is measured, not assumed.

## Load the unchanged baseline and experiment paths

Select the completed stable hybrid run below. Its fixture/environment must match the current generated export and existing index. If the service was deleted, stop: this notebook never redeploys or repopulates it. Do not rerun preparation while measuring.

In [ ]:
import subprocess
import sys
import venv
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.preview import comparison, validate_baseline

if sys.version_info < (3, 11) or version('azure-search-documents') != '12.0.0':
    raise RuntimeError('Use the original demo Python 3.11+ kernel with Search 12.0.0; preview runs in its separate environment.')
reproduction = demo_root / 'outputs' / 'generated' / 'reproduction'
baseline = reproduction / 'runs' / '20261030-live-v2' / 'python-hybrid.json'
manifest_path = reproduction / 'fixture' / 'manifest.json'
env_file = demo_root / 'env' / '.env'
stable = validate_baseline(baseline, manifest_path, env_file)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
run_dir = reproduction / 'preview-runs' / run_id
print(f'Baseline: {baseline}')
print(f'Stable count / unique: {stable["summary"]["reportedCount"]} / {stable["summary"]["uniqueReachableIds"]}')
print(f'New evidence directory: {run_dir}')


## Create the isolated preview environment [PREVIEW]

Install only the pinned preview requirements into `.venv-preview`. This does not upgrade the original `.venv`, install a second notebook kernel, or change the .NET client.

In [ ]:
preview_env = demo_root / '.venv-preview'
preview_python = preview_env / 'Scripts' / 'python.exe'
if not preview_python.exists():
    venv.EnvBuilder(with_pip=True).create(preview_env)
installation = subprocess.run([str(preview_python), '-m', 'pip', 'install', '-r', str(demo_root / 'requirements-preview.txt')], capture_output=True, text=True)
print(installation.stdout)
if installation.returncode:
    raise RuntimeError(f'Preview requirements installation failed ({installation.returncode}):\n{installation.stderr}')
check = subprocess.run([str(preview_python), '-c', "from importlib.metadata import version; assert version('azure-search-documents') == '12.1.0b2'; assert version('azure-identity') == '1.25.3'; print('Verified isolated Search 12.1.0b2 / Identity 1.25.3')"], capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    raise RuntimeError(f'Preview package check failed ({check.returncode}):\n{check.stderr}')
preview_arguments = [str(preview_python), '-u', '-m', 'app.preview', 'run', '--baseline', str(baseline), '--manifest', str(manifest_path), '--env-file', str(env_file), '--run-dir', str(run_dir)]


## Measure retrievable counts with 1,000 text candidates [PREVIEW]

Requests `countRetrievableResults` and an explicit text recall window of 1,000, keeping the same query vector, `k=50`, five-result pages, and no explicit ordering. Every advertised offset and selected repeated boundary is captured. A short response with continuation stops the case as incomplete, never exhausted.

In [ ]:
result = subprocess.run([*preview_arguments, '--scenario', 'retrievable'], cwd=demo_root, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f'Retrievable-count measurement incomplete ({result.returncode}). Evidence is retained; inspect it and continue the independent expanded case only for investigation.')


## Measure retrievable counts with 6,158 text candidates [PREVIEW]

Only `maxTextRecallSize` changes from the preceding preview case. A larger recall window may increase resource usage/latency. We measure unique IDs and duplicates rather than assuming that 6,158 advertised matches guarantees complete unique-ID coverage.

In [ ]:
result = subprocess.run([*preview_arguments, '--scenario', 'expanded'], cwd=demo_root, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f'Expanded-recall measurement incomplete ({result.returncode}). Inspect the retained evidence and comparison; do not overwrite or silently count failures as empty pages.')


## Compare the three observed scenarios

Local recomputation only; no replacement searches. The report distinguishes successful symptom measurements from causal acceptance, carries forward the keyword-baseline caveat, and rejects count alignment if a boundary probe finds results beyond the advertised range. Missing or failed cases remain explicit gaps.

In [ ]:
from IPython.display import FileLink, Markdown, display
finding = comparison(baseline, run_dir)
display(Markdown((run_dir / 'preview-comparison.md').read_text(encoding='utf-8')))
for evidence in sorted(run_dir.glob('*')):
    display(FileLink(str(Path('..') / evidence.relative_to(demo_root))))
display(FileLink(str(Path('..') / baseline.relative_to(demo_root))))
if not finding['measurementsComplete']:
    raise RuntimeError('Preview measurements are incomplete or inconsistent; review the reported gaps, not just the headline counts.')
print(f'Causal acceptance: {finding["causalAcceptance"]}. This is not a production recommendation.')


## Finish

Keep the original and preview evidence separate. For another complete comparison, rerun **Load the unchanged baseline and experiment paths** to create a fresh run directory. Existing per-scenario reports cannot be overwritten. No cleanup is performed here; the Search service remains billable until separately approved deletion.